# Lab view: diffraction geometry with a 4S+2D diffractometer

Shows the sample, the crystal axes, the incident beam $k_i$ (yellow), the scattered beam $k_f$ (orange) and $Q = k_f - k_i$ (magenta)
together with a schematic 4S+2D (You) diffractometer for a set of angles. Crystal axes: reciprocal a\*, b\*, c\* red / green / blue
(default, they match the hkl of orientations and $Q = h a^* + k b^* + l c^*$); `axes="real"` or `"both"` adds the real axes a, b, c in lighter shades.
The triad on the floor shows the **SwissFEL** axes (x dark red, y dark green, z dark blue).

| physical direction | diffcalc | SwissFEL | SWIFT scene |
|---|---|---|---|
| up | +x | +y | +z |
| along beam | +y | +z | +x |
| left, looking downstream | +z | +x | +y |

Lab vectors given by the user (orientations, surface normal) are in the **SwissFEL frame** by default (`frame=` to change).

Set `BROWSER = "headless"` to render without a window and save images with `vis.snapshot(path)`.

In [ ]:
from diffraction import Crystal
from diffraction.visualization.widgets import lab_view_widget

BROWSER = "notebook"   # or None (browser tab) / "headless" (no window, use vis.snapshot)

## Crystal: c-axis normal up, a along the beam (SwissFEL frame)

In [ ]:
crystal = Crystal.from_cif("Bi2Se3", "../diffraction/data/Bi2Se3.cif")
crystal.add_orientation((0, 0, 1), (0, 1, 0), "normal up")       # frame="swissfel" (default)
crystal.add_orientation((1, 0, 0), (0, 0, 1), "a along beam")
crystal.calc_ub()
crystal.set_surface_normal((0, 0, 1))   # reciprocal-lattice vector; used by betain/betaout and the drawing
crystal.set_constraints(mu=0, chi=0, phi=0)
print(crystal)

In [ ]:
energy = 7500
df = crystal.calc_angles((0, 0, 15), energy)
df[["mu", "eta", "chi", "phi", "gamma", "delta", "betain", "betaout"]]

## Show one solution

In [ ]:
vis = crystal.show_lab(df.iloc[0], energy, browser=BROWSER)
print(vis.summary())

The geometry is also available as numbers, in any frame:

In [ ]:
geo = crystal.calc_lab_geometry(df.iloc[0], energy)
{k: v.round(4) for k, v in geo.in_frame("swissfel").items() if k in ("k_i", "k_f", "Q", "surface_normal")}

## Interactive controls

Pick a `calc_angles` solution for any hkl, then move the angles with the sliders (updates on release).

In [ ]:
vis, controls = lab_view_widget(crystal, energy, hkl=(1, 0, 10), vis=vis)
controls

## Lab and reciprocal space together

One set of controls can drive several views: here the lab view above and the Ewald construction
(see `visualization_reciprocal_space.ipynb`).

In [ ]:
from diffraction.visualization.widgets import angle_controls

rvis = crystal.show_reciprocal(energy=energy, browser=BROWSER)
both = angle_controls(crystal, energy, [vis, rvis], hkl=(0, 1, 5))
both

## Snapshots (headless mode only)

In [ ]:
if BROWSER == "headless":
    h = vis.origin[2]
    vis.snapshot("lab_view_side.png", position=[0.2, 2.2, h + 0.3], look_at=[0.2, 0, h])

In [ ]:
vis.close()
rvis.close()